In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module_DM import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# The demo mongod instance has no authentication
username = ""
password = ""

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# Server-side pagination: only ever fetch one page worth of documents at a
# time, rather than the full matching result set, so this scales to a much
# larger collection than 10,000 documents.
PAGE_SIZE = 15

# class read method must support return of list object and accept projection json input
# this initial load is just page 1 of the unfiltered view - enough to seed
# the table's column list and its first page of data
df = pd.DataFrame.from_records(db.read({}, limit=PAGE_SIZE))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), style = {"height" : '200px'}),
    html.Center(html.B(html.H1('Grazioso Salvare Dashboard CS-340 Dashboard - Dylan Mousseau'))),
    html.Hr(),
    html.Div(
        
    # Filter radio buttons
        dcc.RadioItems(
            id='filter-type',
            options = [
                {'label' : 'Water Rescue', 'value' : 'water'},
                {'label' : 'Mountain/Wilderness Rescue', 'value' : 'mountain'},
                {'label' : 'Disaster Rescue & Individual Tracking', 'value' : 'disaster'},
                {'label' : 'Reset', 'value' : 'reset'}
            ],
            value = 'reset'
        )
    ),
    html.Div(id='page-count-id'),
    html.Hr(),
    dash_table.DataTable(        
        id='datatable-id',
        # rescue_category drives filtering but is an array field, which
        # renders as a raw Python-list string, so it is left out of the
        # visible columns.
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns if i != "rescue_category"],
        data=df.to_dict('records'),
        editable = False, # editable flag for table
        # native filter/sort need the complete matching result set loaded
        # Pages are returned in a fixed rec_num order instead (see
        # update_dashboard) so paging is at least stable and predictable.
        filter_action = "none",
        sort_action = "none",
        sort_mode = "multi", # multi-column sorting option
        column_selectable = "single", # single column radio button selection
        row_selectable = "single", # single row radio button selection
        row_deletable = False, # delete row parameter set to False
        selected_columns = [], # IDs of selected columns in UI
        selected_rows = [], # IDs of selected rows in UI
        page_action = 'custom', # server-side pagination - only page_size rows are ever sent to the client
        page_current = 0, # current page user is on
        page_size = PAGE_SIZE, # row limit per page 
    ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
# Both callbacks below need "what query does this filter mean" - shared
# here so the logic only lives in one place.
def query_for_filter(filter_type):
    if filter_type in ('water', 'mountain', 'disaster'):
        return {"rescue_category" : filter_type}
    return {}

@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value'),
               Input('datatable-id', 'page_current'),
               Input('datatable-id', 'page_size')])
def update_dashboard(filter_type, page_current, page_size):
# Filter for rescue scenario using the precomputed rescue_category field
# (see the rescue_category migration) instead of rebuilding a compound
# breed/sex/age query on every callback. Only this one page worth of
# documents is fetched from MongoDB - not the full matching result set.
    query = query_for_filter(filter_type)
    page_current = page_current or 0
    page_size = page_size or PAGE_SIZE
    df = pd.DataFrame.from_records(
        db.read(query, sort=[("rec_num", 1)], skip=page_current * page_size, limit=page_size)
    )

    if not df.empty:
        df.drop(columns=['_id'],inplace=True)

    return df.to_dict('records')

# Resets to page 1 whenever the filter changes
# and reports how many records/pages the current filter has, using
# count_documents rather than fetching the documents just to count them.
@app.callback(
    [Output('datatable-id', 'page_current'),
     Output('page-count-id', 'children')],
    [Input('filter-type', 'value')])
def reset_page_on_filter_change(filter_type):
    query = query_for_filter(filter_type)
    total = db.count(query)
    pages = max(1, -(-total // PAGE_SIZE))  # ceiling division
    return 0, f"{total} records, {pages} pages"

# Display the breeds of animal based on quantity represented in
# the data table
# Breed counts are computed server-side with an aggregation pipeline
# ($match + $group), so only a handful of {breed, count} rows come back
# instead of pulling every matching document to build the chart from.
@app.callback(
    Output('graph-id', "children"),
    [Input('filter-type', 'value')])
def update_graphs(filter_type):
    if filter_type not in ('water', 'mountain', 'disaster'):
        return []

    pipeline = [
        {'$match': {'rescue_category': filter_type}},
        {'$group': {'_id': '$breed', 'count': {'$sum': 1}}},
        {'$sort': {'count': -1}},
    ]
    agg_df = pd.DataFrame.from_records(db.aggregate(pipeline))
    agg_df.rename(columns={'_id': 'breed'}, inplace=True)

    return [
        dcc.Graph(            
            figure = px.pie(agg_df, names='breed', values='count', title='Preferred Animals')
        )    
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    # derived_virtual_selected_rows defaults to [], not None, when nothing
    # is selected (e.g. right after switching filters) - index[0] on an
    # empty list throws IndexError, so both cases must be checked here.
    elif not index:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Referenced by column name rather than position, so this
            # keeps working if the schema's column order changes.
            dl.Marker(position=[dff.iloc[row]['location_lat'],dff.iloc[row]['location_long']], children=[
                dl.Tooltip(dff.iloc[row]['breed']),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row]['name'])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode
app.run_server() 